## Импорты

In [ ]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

In [ ]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
from models.trainer import train_model
from models.rut5_paraphraser.model import ParaphraserModel

config = load_config("config.yaml")

## Загрузка датасета

In [ ]:
import os
if not os.path.exists("data/paraphrases_clean.parquet"):
    !wget -O data/paraphrases_clean.parquet https://github.com/AbonentVneSeti/text_processing_final_project/main/data/paraphrases_clean.parquet

df = pd.read_parquet("data/paraphrases_clean.parquet")
print(f"Загружено {len(df)} пар")

In [ ]:
train_loader, val_loader = build_dataloaders(df, config["model_config"])

## Обучение модели

In [ ]:
model = ParaphraserModel(config["model_config"])
trained_model = train_model(
    model, train_loader, val_loader,
    config["model_config"], config["trainer_config"], config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

## Сохранение (для Colab)

In [ ]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r models/rut5_paraphraser/saves /content/drive/MyDrive/
    print("Модель сохранена на Google Drive")

In [ ]:
if is_colab:
    google.colab.runtime.unassign()